# Quickstart: the three endpoints on one page

## Overview

This page is the reference. It shows every endpoint your team can call, the exact URL, the JSON you send, and the fields you get back, each with the shortest working code. Read it first, keep it open while you build, and go to Labs 1 to 4 when you want the details behind a feature.

One base URL, one API key per team, three operations:

| Operation | Path | Model name |
|---|---|---|
| Chat | `POST /v1/chat/completions` | `hackathon-chat` |
| Embeddings | `POST /v1/embeddings` | `hackathon-embedding` |
| Images | `POST /v1/images/generations` | `hackathon-image` |

Every request carries two headers: `Authorization: Bearer <your team key>` and `Content-Type: application/json`. Nothing else in the OpenAI API exists here; other paths return `404`.

## Prerequisites

- Python 3.10 or newer
- Your team's API key in the `.env` file at the repository root (copy `.env.example`)
- About 10 minutes

## Step 1: Set up your environment

In [ ]:
# Installs the packages this page needs. Safe to rerun.
%pip install -q "openai>=3,<4" python-dotenv requests

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Reads OPENAI_BASE_URL and OPENAI_API_KEY from the .env file in the repository root.
# override=True makes the file win over any OPENAI_API_KEY already set in your shell.
load_dotenv(Path("..") / ".env", override=True)
load_dotenv(override=True)

base_url = os.environ["OPENAI_BASE_URL"]
api_key = os.environ["OPENAI_API_KEY"]
print("Endpoint:", base_url)
print("Key:     ", api_key[:6] + "..." + api_key[-4:])

client = OpenAI()  # The SDK reads the same two variables, so no arguments are needed.

## Step 2: What a call looks like on the wire

Before any library, here is the raw HTTP request. This is what every SDK and framework sends underneath. If you can make this call, you can use the endpoint from any language.

```bash
curl https://ws.akamai-workshops.com/v1/chat/completions \
  -H "Authorization: Bearer $OPENAI_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{"model": "hackathon-chat", "messages": [{"role": "user", "content": "Say hello."}], "max_tokens": 20}'
```

The same request from Python with `requests`, so you can see the status code, the headers, and the full JSON body:

In [ ]:
import json
import requests

url = f"{base_url}/chat/completions"
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
body = {"model": "hackathon-chat", "messages": [{"role": "user", "content": "Say hello."}], "max_tokens": 20}

r = requests.post(url, headers=headers, json=body, timeout=60)
print("Status:", r.status_code)
print("Rate-limit headers:", {k: v for k, v in r.headers.items() if k.lower().startswith("x-ratelimit")})
print(json.dumps(r.json(), indent=2)[:900])

From here on the `openai` package does the HTTP for you. `client.chat.completions.create(...)` sends exactly the body above and returns the JSON as a Python object.

## Step 3: Chat

`POST /v1/chat/completions`

| Field | Required | Values and defaults |
|---|---|---|
| `model` | yes | `"hackathon-chat"` |
| `messages` | yes | list of `{"role": "system" or "user" or "assistant", "content": "..."}` |
| `max_tokens` | no | output limit; default 512, maximum 4096 |
| `temperature` | no | 0 for repeatable answers; model default when omitted |
| `stream` | no | `true` to receive the answer as it is generated |
| `response_format` | no | `{"type": "json_schema", "json_schema": {...}}` to force a JSON shape |
| `tools`, `tool_choice` | no | function definitions the model may call |
| `extra_body` | no | `{"chat_template_kwargs": {"enable_thinking": true}}` turns thinking on |

Response: `choices[0].message.content` is the answer, `choices[0].finish_reason` is `stop` or `length`, `usage` has the token counts.

In [ ]:
response = client.chat.completions.create(
    model="hackathon-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Keep answers short."},
        {"role": "user", "content": "What is an API key?"},
    ],
    max_tokens=80,
    temperature=0,
)
print(response.choices[0].message.content)
print("\nfinish_reason:", response.choices[0].finish_reason, "| tokens:", response.usage.total_tokens)

Streaming is the same call with `stream=True`. You get chunks; print each `delta.content` as it arrives.

In [ ]:
for chunk in client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Count from one to five, one number per line."}],
    max_tokens=40,
    stream=True,
):
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

## Step 4: Embeddings

`POST /v1/embeddings`

| Field | Required | Values and defaults |
|---|---|---|
| `model` | yes | `"hackathon-embedding"` |
| `input` | yes | one string, or a list of up to 16 strings |
| `encoding_format` | no | `"float"` or `"base64"`; the SDK handles this for you |

Response: `data[i].embedding` is a list of 1,024 floats for input `i`, in order. `usage.prompt_tokens` counts the input.

In [ ]:
response = client.embeddings.create(
    model="hackathon-embedding",
    input=["The cat sat on the mat.", "A feline rested on a rug.", "Quarterly revenue grew 8 percent."],
)
vectors = [item.embedding for item in response.data]
print(len(vectors), "vectors, each", len(vectors[0]), "numbers")

# Similar meaning gives similar vectors. Compare the first text to the other two.
import math
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))
print("cat vs feline:  ", round(cosine(vectors[0], vectors[1]), 3))
print("cat vs revenue: ", round(cosine(vectors[0], vectors[2]), 3))

## Step 5: Images

`POST /v1/images/generations`

| Field | Required | Values and defaults |
|---|---|---|
| `model` | yes | `"hackathon-image"` |
| `prompt` | yes | text, up to 2,000 characters |
| `size` | no | `"512x512"`, `"768x768"`, or `"1024x1024"`; default 1024 |
| `n` | no | always 1 |
| `response_format` | no | always `"b64_json"` |
| `extra_body` | no | `{"seed": 7}` for a different or repeatable result; default seed 0 |

Response: `data[0].b64_json` is the PNG as a base64 string. Decode it and you have a file.

In [ ]:
import base64
from IPython.display import Image as ShowImage

response = client.images.generate(
    model="hackathon-image",
    prompt="A yellow bicycle leaning on a brick wall, simple illustration",
    size="512x512",
    response_format="b64_json",
    extra_body={"seed": 3},
)
png_bytes = base64.b64decode(response.data[0].b64_json)
with open("quickstart.png", "wb") as f:
    f.write(png_bytes)
print("Saved quickstart.png,", len(png_bytes), "bytes")
ShowImage(data=png_bytes)

## Step 6: Limits and errors you will meet

| Limit, per team per minute | Value |
|---|---|
| Requests, all endpoints | 60 |
| Chat tokens, input plus output | 100,000 |
| Image requests | 6 |

| Status | Meaning | What to do |
|---|---|---|
| `401` | wrong or missing key | fix `.env`; the key starts with `hk_` |
| `404` | path does not exist | only the three paths above work |
| `422` | a field is not allowed | check the tables: 16 embedding inputs, square image sizes, `b64_json` |
| `429` | quota or busy queue | wait the seconds in `x-ratelimit-reset` or `Retry-After`, then retry |
| `finish_reason: length` | answer was cut off | raise `max_tokens`, up to 4096 |

## Validation

In [ ]:
assert response.data[0].b64_json and len(vectors[0]) == 1024
print("Quickstart complete. Your key, the endpoint, and all three operations work.")

## Clean up

In [ ]:
from pathlib import Path
Path("quickstart.png").unlink(missing_ok=True)
print("Removed quickstart.png")

## Next steps

- Lab 1 goes deep on chat: conversations, long documents, thinking mode, quotas.
- Lab 2 builds semantic search and retrieval with embeddings.
- Lab 3 covers image sizes, seeds, and the image queue.
- Lab 4 adds tool calling and structured JSON output.